# Updating Plasma and Convergence

As light travels through a real plasma, it has effects on the properties of the plasma due to light-matter
interactions as well as the presence of extra energy from the light. Additionally, as [previously discussed](../montecarlo/propagation.rst), properties of the plasma affect how light travels through it. This is a typical equilibrium problem. We solve for the plasma properties by finding a steady-state solution; that is, the actual plasma will be in a state such that the plasma state will not change as
light propagates through it, because the effects of the light on the plasma and the effects of the plasma on the
light are in equilibrium.

One of TARDIS's main goals is to determine this plasma state (as we need the actual plasma properties in order to
get an accurate spectrum). This is done in an iterative process. After each [Monte Carlo iteration](../montecarlo/index.rst) (which sends light through the supernova ejecta), TARDIS uses the [Monte Carlo estimators](../montecarlo/estimators.rst)
how the propagating light affects the plasma state, after which the plasma state is updated (as will be explained below and demonstrated in the code example). We do this many times, and attempt to have the plasma state converge
to the steady-state we are looking for. In fact, all but the last Monte Carlo iteration is used for this purpose
(after which TARDIS will have the necessary plasma state for its last iteration which calculates the spectrum).

<div class="alert alert-info">

Note

For all but the last iteration, TARDIS uses the number of Monte Carlo packets specified in the
[Monte Carlo configuration](../../io/configuration/components/montecarlo.rst) under the ``no_of_packets`` argument. This is because
a different number of packets may be necessary to calculate the spectrum as opposed to calculate the
plasma state.

</div>

After each iteration the values for radiative temperature and dilution factor are updated by calling the ``advance_state`` method on a ``Simulation`` object as will be shown below in the code example. The goal of this is to eventually have the radiative temperature and dilution factor converge to a single value so that the steady-state plasma state can be determined. To ensure that the simulation converges, TARDIS employs a convergence strategy. Currently, only one convergence strategy is available: damped convergence. This will be described in the following sections.

<div class="alert alert-info">

Note
    
Unless otherwise noted, all user-supplied quantities mentioned on this page are supplied in the [convergence section of the Monte Carlo configuration](../../io/configuration/components/montecarlo.rst#damped-convergence-strategy), which will be referenced as the convergence configuration.

</div>

## $T_\mathrm{rad}$ and $W$

The main way in which the plasma state is updated is by using the ``j`` and ``nu_bar`` estimators (see [here](../montecarlo/estimators.rst#j-and-bar-nu-estimators)) to update two of the key plasma inputs (see [Plasma](../setup/plasma/index.rst)): the radiative temperature $T_\mathrm{rad}$ and dilution factor $W$. Recall that each of these quantities takes on a different value in each shell.

Using the estimators $J$ and $\bar \nu$, we estimate these quantities using

$$T_{\mathrm{rad\ estimated}} = \frac{h}{k_{\mathrm{B}}} \frac{\pi^4}{360 \zeta(5)} \frac{\bar \nu}{J}$$

and

$$W_\mathrm{estimated} = \frac{\pi J}{\sigma_{\mathrm{R}} T_{\mathrm{rad\ estimated}}^4}$$

where $h$ is Planck's constant, $k_{\mathrm{B}}$ is Boltzmann's constant, $\sigma_{\mathrm{R}}$ is the Stefan–Boltzmann constant, and $\zeta$ is the Riemann zeta function. The equation for $W$ comes from the fact that the dilution factor is the ratio of the actual mean intensity to that of a blackbody, which is $J_{\mathrm{blackbody}}=\frac{\sigma_{\mathrm{R}} T^4}{\pi}$.

Recall (see [Implementation](../montecarlo/estimators.rst#implementation)), however, that when TARDIS stores these estimators, it leaves out the prefactor of $\frac{1}{4\pi V\Delta t}$ for computational ease. That is, $J=\frac{1}{4\pi V\Delta t}\sum_i \varepsilon_i l_i=\frac{1}{4\pi V\Delta t}*\mathrm{real\ j\ estimator}$, and $\bar \nu=\frac{1}{4\pi V\Delta t}\sum_i \varepsilon_i \nu_i l_i=\frac{1}{4\pi V\Delta t}*\mathrm{real\ nu\_ bar\ estimator}$. These factors are then included in our calculations; specifically, using the previous relations we have

$$T_\mathrm{rad\ estimated}= \frac{h}{k_{\mathrm{B}}} \frac{\pi^4}{360 \zeta(5)} \frac{\sum_i \varepsilon_i \nu_i l_i}{\sum_i \varepsilon_i l_i} = \frac{h}{k_{\mathrm{B}}} \frac{\pi^4}{360 \zeta(5)} \frac{\mathrm{real\ nu\_ bar\ estimator}}{\mathrm{real\ j\ estimator}}$$
and
$$W_\mathrm{estimated} = \frac{\sum_i \varepsilon_i l_i}{4\sigma_{\mathrm{R}} T_{\mathrm{rad\ estimated}}^4V\Delta t} = \frac{\mathrm{real\ j\ estimator}}{4\sigma_{\mathrm{R}} T_{\mathrm{rad\ estimated}}^4V\Delta t}.$$

While TARDIS can then update the plasma state using the estimated values, there is a good chance that these estimated values would “overshoot” the true value we want to converge to (for example, if the current value of the dilution factor in some cell the dilution factor is 0.4, and the true steady-state value TARDIS wants to find is 0.45, there is a good chance that the estimated value will be greater than 0.45). This could make the simulation take longer to converge or, at worst, make it so the simulation does not converge at all. To account for this, users can set (in the convergence configuration) a "damping constant" for both the radiative temperature ($d_{T_\mathrm{rad}}$) and the dilution factor ($d_W$). When ``advance_state`` is called, these quantities update as follows:

$$T_\mathrm{rad\ updated} = T_\mathrm{rad\ current} + d_{T_\mathrm{rad}}(T_\mathrm{rad\ estimated}-T_\mathrm{rad\ current})$$
    
and
    
$$ W_\mathrm{updated} = W_\mathrm{current} + d_W(W_\mathrm{estimated}-W_\mathrm{current}).$$

This means, for example, if the damping constant is 0.5, the updated value is halfway between the current value and the estimated value. If the damping constant is 0.7, the updated value is 70% of the way between the current value and the estimated value, and so on. **If the damping constant is 1, then the updated value is exactly the estimated value, and if the damping constant is zero, the value stays the same throughout the simulation and is not updated.**

The updated $T_\mathrm{rad}$ and $W$ are then used as inputs to the updated [plasma calculations](../setup/plasma/index.rst) which account for the effect of the Monte Carlo packets on the plasma state.

## $T_\mathrm{inner}$

The temperature of the inner boundary, $T_\mathrm{inner}$, plays a unique role in the simulation, as it is the primary determiner of the output luminosity. This is because the the luminosity of the inner boundary is proportional to $T_\mathrm{inner}^4$ (see [Energy Packet Initialization](../montecarlo/initialization.ipynb)). Thus, $T_\mathrm{inner}$ is updated throughout the simulation in order to match the output luminosity to the requested luminosity specified in the [supernova configuration](../../io/configuration/components/supernova.rst) between the bounds specified in the supernova configuration. However, there is not necessarily a quartic relationship between $T_\mathrm{inner}$ and the output luminosity, as changing $T_\mathrm{inner}$ also changes the frequency distribution of the initialized packets (once again see [Energy Packet Initialization](../montecarlo/initialization.ipynb)). This then affects the light-matter interactions, affecting which packets make it to the outer boundary, which also affects the output luminosity. Because of this, there is not an exact way to estimate $T_\mathrm{inner}$. To do this estimation, we use

$$T_\mathrm{inner\ estimated} = T_\mathrm{inner\ current} * \left(\frac{L_\mathrm{output}}{L_\mathrm{requested}}\right)^{\mathrm{t\_inner\_update\_exponent}}$$
    
where $L_\mathrm{output}$ is the output luminosity calculated by adding up the luminosity of each packet (see [Basic Spectrum Generation](../spectrum/basic.ipynb)) between the bounds specified in the [supernova configuration](../../io/configuration/components/supernova.rst), $L_\mathrm{requested}$ is the luminosity requested also in the supernova configuration (requested between those bounds previously mentioned), and ``t_inner_update_exponent`` is provided by the user in the convergence configuration. Note that what we are doing is "correcting" the previous value of the inner temperature by a factor of $\left(\frac{L_\mathrm{output}}{L_\mathrm{requested}}\right)^{\mathrm{t\_inner\_update\_exponent}}$. Note that if $\frac{L_\mathrm{output}}{L_\mathrm{requested}}$ is greater than 1, we want to lower $T_\mathrm{inner}$ as the output luminosity is too high, and vice versa if the ratio is less than 1. Thus ``t_inner_update_exponent`` should be negative. Naively one might set ``t_inner_update_exponent=-0.25``, however as a default TARDIS uses ``t_inner_update_exponent=-0.5`` as -0.25 may undershoot the correct $T_\mathrm{inner}$ because of its previously mentioned effects on the initial frequency distribution.

After calculating the estimated $T_\mathrm{inner}$, the quantity is updated using damped convergence with its own damping constant (once again set in the convergence configuration):

$$T_\mathrm{inner\ updated} = T_\mathrm{inner\ current} + d_{T_\mathrm{inner}}(T_\mathrm{inner\ estimated}-T_\mathrm{inner\ current}).$$

Once again, If the damping constant is 1, then the updated value is exactly the estimated value, and if the damping constant is zero, the value stays the same throughout the simulation and is not updated.

Additionally, because of the vast impact of $T_\mathrm{inner}$ on the simulation, one may want to update it less frequently -- i.e. allow $W$ and $T_\mathrm{rad}$ to reach a steady-state value for a particular $T_\mathrm{inner}$ before updating $T_\mathrm{inner}$. To do this, in the convergence configuration we set ``lock_t_inner_cycles``, which is the number of iterations to wait before updating $T_\mathrm{inner}$. It is set to 1 by default, meaning $T_\mathrm{inner}$ would be updated every iteration.

## Convergence Information

During the simulation, information about the how $T_\mathrm{rad}$, $W$, and $T_\mathrm{inner}$ are updated as well as a comparison of the total output luminosity and the requested luminosity are logged at the INFO level (see [Configuring the Logging Output for TARDIS](../../io/optional/tutorial_logging_configuration.ipynb)) as shown in the code below, to give users a better idea of how the convergence process is working.

In addition, TARDIS allows for the displaying of convergence plots, which allows users to visualize the convergence process for $T_\mathrm{rad}$, $W$, $T_\mathrm{inner}$, and the total luminosity of the supernova being modeled. For more information, see [Convergence Plots](../../io/visualization/tutorial_convergence_plot.ipynb).

## Convergence Criteria

TARDIS also allows users to stop the simulation if the simulation reaches a certain level of convergence, which is checked upon the call of the ``advance_state`` method. To enable this, users must set ``stop_if_converged=True`` in the convergence configuration. Also in the configuration, the quantities ``hold_iterations``, ``threshold``, and ``fraction`` are be specified to determine convergence as follows:

For the simulation to be considered to have converged, for ``hold_iterations`` successive iterations, the estimated values of $T_\mathrm{rad}$, $W$, and $T_\mathrm{inner}$ may differ from the previous value by a fraction of at most ``threshold`` in at least ``fraction`` fraction of the shells (for $T_\mathrm{inner}$, since there is only one value, the ``fraction`` part does not apply). For example, if ``hold_iterations=3``, ``threshold=0.05`` for all three quantities, and ``fraction=0.8``, the simulation will be considered to have converged if for 3 successive iterations the estimated values of $T_\mathrm{rad}$ and $W$ differ from the current respective values by at most 5% in at least 80% of the shells, *and* the estimated $T_\mathrm{inner}$ differs by at most 5%. See the [convergence configuration schema](../../io/configuration/components/montecarlo.rst#damped-convergence-strategy) for default values of these quantities.

<div class="alert alert-info">

Note
    
To determine convergence, we compare the estimated value, **not** the updated value (which is related to the estimated value via the damping constant), with the previous value. If $T_\mathrm{inner}$ is locked (see the previous section), the estimated value will still be calculated so convergence can be checked as usual.

</div>

<div class="alert alert-info">

Note
    
``hold_iterations`` and ``fraction`` are universal quantities, i.e. they are each a single value that applies to $T_\mathrm{rad}$ and $W$, and for ``hold_iterations`` also $T_\mathrm{inner}$. ``threshold``, on the other hand, is supplied for each quantity separately, so for instance you could require $T_\mathrm{rad}$ to differ by less than 1%, $W$ to differ by less than 3%, and $T_\mathrm{inner}$ to differ by less than 5% for convergence to be reached.

</div>


## Updating Other Quantities

Other quantities in the plasma state can depend on other estimators. Currently, this is only implemented for the ``j_blue`` estimator: If ``radiative_rates_type`` in the [plasma configuration](../../io/configuration/components/plasma.rst) is set to ``detailed``, the `j_blues` plasma property will will be replaced with the value of the $J^b_{lu}$ [estimator](../montecarlo/estimators.rst#j-b-lu-estimator) (the raw estimator times the factor of $\frac{ct_\mathrm{explosion}}{4\pi V \Delta t}$, once again see [Implementation](../montecarlo/estimators.rst#implementation)), which would then affect other plasma properties that depend on the `j_blues` values (see [Plasma](../setup/plasma/index.rst)). Otherwise, the `j_blues` values in the plasma are calculated as they typically are in the plasma calculations, and the $J^b_{lu}$ estimator is only used for the [formal integral](../spectrum/formal_integral.rst). Even in the former case, while the estimator does contribute to the updating of the plasma when the ``advance_state`` method is called, it does **not** contribute to the determination of if the simulation has converged, and it does **not** show up in the convergence logging or convergence plots.

## Code Example

We now show a detailed example of how the plasma is updated using the estimators after a Monte Carlo iteration. First, we import the necessary packages and set up a simulation (see [Setting Up the Simulation](../setup/index.rst)):

In [1]:
from tardis.io.configuration.config_reader import Configuration
from tardis.simulation import Simulation
from tardis.io.atom_data import download_atom_data
import numpy as np
from scipy.special import zeta

from astropy import units as u, constants as const

# We download the atomic data needed to run this notebook
download_atom_data('kurucz_cd23_chianti_H_He_latest')

Iterations:          0/? [00:00<?, ?it/s]

Packets:             0/? [00:00<?, ?it/s]

Atomic Data kurucz_cd23_chianti_H_He_latest already exists in /home/runner/Downloads/tardis-data/kurucz_cd23_chianti_H_He_latest.h5. Will not download - override with force_download=True.


In [2]:
tardis_config = Configuration.from_yaml('tardis_example.yml')

# We manually put in the damping constants and t_inner_update_exponent for
# illustrative purposes:
damping_t_radiative = 0.5
damping_dilution_factor = 0.3
damping_t_inner = 0.7
t_inner_update_exponent = -0.5

# We set the above values into the configuration:
tardis_config.montecarlo.convergence_strategy.t_rad.damping_constant = damping_t_radiative
tardis_config.montecarlo.convergence_strategy.w.damping_constant = damping_dilution_factor
tardis_config.montecarlo.convergence_strategy.t_inner.damping_constant = damping_t_inner
tardis_config.montecarlo.convergence_strategy.t_inner_update_exponent = t_inner_update_exponent

In [3]:
sim = Simulation.from_config(tardis_config)

simulation_state = sim.simulation_state
plasma = sim.plasma
transport = sim.transport

Number of density points larger than number of shells. Assuming inner point irrelevant


model_isotope_time_0 is not set in the configuration. Isotopic mass fractions will not be decayed and is assumed to be correct for the time_explosion. THIS IS NOT RECOMMENDED!


We show the initial radiative temperature and dilution factor in each shell, the initial inner boundary temperature, and the initial electron densities in each shell:

In [4]:
simulation_state.t_radiative

<Quantity [9926.50196546, 9911.63537753, 9896.81325339, 9882.03539385,
           9867.30160093, 9852.6116778 , 9837.96542884, 9823.36265956,
           9808.80317663, 9794.28678787, 9779.81330223, 9765.3825298 ,
           9750.99428177, 9736.64837045, 9722.34460927, 9708.08281272,
           9693.8627964 , 9679.68437699, 9665.54737223, 9651.45160094] K>

In [5]:
simulation_state.dilution_factor

array([0.40039164, 0.33245223, 0.28966798, 0.2577141 , 0.23224568,
       0.21120466, 0.19341188, 0.17811402, 0.16479446, 0.1530809 ,
       0.14269498, 0.13342262, 0.12509541, 0.11757849, 0.11076215,
       0.10455605, 0.09888494, 0.09368554, 0.08890421, 0.08449514])

In [6]:
simulation_state.t_inner

<Quantity 9933.95199592 K>

In [7]:
plasma.electron_densities

0     2.865134e+09
1     2.182365e+09
2     1.678840e+09
3     1.303472e+09
4     1.020811e+09
5     8.059447e+08
6     6.411609e+08
7     5.137297e+08
8     4.144079e+08
9     3.364195e+08
10    2.747519e+08
11    2.256656e+08
12    1.863476e+08
13    1.546660e+08
14    1.289928e+08
15    1.080764e+08
16    9.094799e+07
17    7.685317e+07
18    6.520063e+07
19    5.552442e+07
dtype: float64

We set the number of packets and we run one iteration of the Monte Carlo simulation:

In [8]:
N_packets = 10000

# Using the commented out code below, we can also get the number of packets
# from the configuration -- try it out:
#N_packets = tardis_config.montecarlo.no_of_packets

sim.iterate(N_packets)

/home/runner/work/tardis/tardis/tardis/transport/montecarlo/montecarlo_main_loop.py:146: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  vpacket_collection = vpacket_collections[i]


(<Quantity 8.0276259e+42 erg / s>,
 array([0., 0., 0., ..., 0., 0., 0.], shape=(10001,)))

We now show the values of the $J$ and $\bar \nu$ estimators, attaching their proper units (note that these are the raw estimators, and the factors of $\frac{1}{4\pi V \Delta t}$ etc are not included):

In [9]:
j_estimator = transport.transport_state.j_estimator * (u.erg * u.cm) 
j_estimator

<Quantity [1.14042102e+14, 9.77570571e+13, 8.76478738e+13, 7.93207787e+13,
           7.38091609e+13, 6.74255572e+13, 6.46529514e+13, 6.07556649e+13,
           5.72185999e+13, 5.52658812e+13, 5.26639359e+13, 5.09976983e+13,
           4.97165152e+13, 4.81900114e+13, 4.74242045e+13, 4.61381909e+13,
           4.51442279e+13, 4.41943087e+13, 4.34918463e+13, 4.24727872e+13] cm erg>

In [10]:
nu_bar_estimator = transport.transport_state.nu_bar_estimator * (u.erg * u.cm * u.Hz)
nu_bar_estimator

<Quantity [9.27543588e+28, 7.95244557e+28, 7.19998419e+28, 6.54419999e+28,
           6.06969337e+28, 5.51634276e+28, 5.29540521e+28, 4.95271779e+28,
           4.64647701e+28, 4.48720297e+28, 4.25148060e+28, 4.09100867e+28,
           3.99038545e+28, 3.82020293e+28, 3.78459229e+28, 3.65066050e+28,
           3.52813380e+28, 3.42286614e+28, 3.36382528e+28, 3.25341531e+28] Hz cm erg>

We show the values of $J$ and $\bar \nu$ including the prefactor:

In [11]:
V = simulation_state.volume
Delta_t = transport.transport_state.time_of_simulation
prefactor = 1 / (4 * np.pi * V * Delta_t)
J = prefactor * j_estimator
J

<Quantity [9.52006929e+10, 7.54373338e+10, 6.27095663e+10, 5.27631107e+10,
           4.57633029e+10, 3.90600136e+10, 3.50724341e+10, 3.09272539e+10,
           2.73854721e+10, 2.49154656e+10, 2.24031003e+10, 2.05040541e+10,
           1.89215457e+10, 1.73866624e+10, 1.62429869e+10, 1.50212417e+10,
           1.39885082e+10, 1.30490113e+10, 1.22505326e+10, 1.14252257e+10] erg / (s cm2)>

In [12]:
nu_bar = prefactor * nu_bar_estimator
nu_bar

<Quantity [7.74299933e+25, 6.13675686e+25, 5.15138436e+25, 4.35311345e+25,
           3.76334337e+25, 3.19564913e+25, 2.87261055e+25, 2.52114698e+25,
           2.22385670e+25, 2.02296152e+25, 1.80856870e+25, 1.64482449e+25,
           1.51869576e+25, 1.37830594e+25, 1.29623857e+25, 1.18854799e+25,
           1.09323674e+25, 1.01065093e+25, 9.47502915e+24, 8.75172241e+24] Hz erg / (s cm2)>

As mentioned [here](../montecarlo/estimators.rst#j-and-bar-nu-estimators), $\bar \nu$ is not truly the mean frequency (as you can see by units). We show the true mean frequency, which will, as expected, be in units of Hz.

In [13]:
nu_bar/J

<Quantity [8.13334346e+14, 8.13490689e+14, 8.21467068e+14, 8.25029721e+14,
           8.22349597e+14, 8.18138253e+14, 8.19050807e+14, 8.15186173e+14,
           8.12057097e+14, 8.11930050e+14, 8.07285010e+14, 8.02194768e+14,
           8.02627745e+14, 7.92737503e+14, 7.98029684e+14, 7.91244830e+14,
           7.81524895e+14, 7.74503832e+14, 7.73438142e+14, 7.65999955e+14] Hz>

We show the calculations of the estimated and updated $T_\mathrm{rad}$ and $W$. Note that the ``decompose()`` method is used to simplify the units:

In [14]:
t_rad_estimated = ( (const.h.cgs / const.k_B.cgs) 
                   * (np.pi**4 / (360 * zeta(5, 1)))
                   * (nu_bar_estimator / j_estimator) ).decompose()
t_rad_estimated

<Quantity [10185.69092159, 10187.64885943, 10287.54004608, 10332.1564818 ,
           10298.59228428, 10245.85205532, 10257.28030871, 10208.88204825,
           10169.69545865, 10168.10439951, 10109.93281472, 10046.18581471,
           10051.60814705,  9927.7489347 ,  9994.02490732,  9909.0556289 ,
            9787.32923708,  9699.40183951,  9686.05580201,  9592.90460195] K>

In [15]:
t_rad_updated = simulation_state.t_radiative + damping_t_radiative * (t_rad_estimated - simulation_state.t_radiative)
t_rad_updated

<Quantity [10056.09644353, 10049.64211848, 10092.17664974, 10107.09593783,
           10082.9469426 , 10049.23186656, 10047.62286877, 10016.1223539 ,
            9989.24931764,  9981.19559369,  9944.87305848,  9905.78417225,
            9901.30121441,  9832.19865258,  9858.18475829,  9808.56922081,
            9740.59601674,  9689.54310825,  9675.80158712,  9622.17810144] K>

In [16]:
dilution_factor_estimated = ( j_estimator / (4 * const.sigma_sb.cgs * t_rad_estimated**4 * V * Delta_t) ).decompose()
dilution_factor_estimated

<Quantity [0.49002342, 0.38799771, 0.31018897, 0.25651053, 0.22539517,
           0.19637164, 0.17553985, 0.15774926, 0.14184928, 0.12913612,
           0.11881021, 0.11152536, 0.1026959 , 0.09916349, 0.09020755,
           0.0863208 , 0.08446044, 0.08168391, 0.07710913, 0.07474859]>

In [17]:
dilution_factor_updated = simulation_state.dilution_factor + damping_dilution_factor * (dilution_factor_estimated - simulation_state.dilution_factor)
dilution_factor_updated

<Quantity [0.42728117, 0.34911588, 0.29582428, 0.25735303, 0.23019053,
           0.20675475, 0.18805027, 0.17200459, 0.15791091, 0.14589747,
           0.13552955, 0.12685344, 0.11837556, 0.11205399, 0.10459577,
           0.09908548, 0.09455759, 0.09008505, 0.08536569, 0.08157118]>

We show the output and requested luminosities, and use these to calculate the estimated and updated $T_\mathrm{inner}$:

In [18]:
# The output luminosity is calculated between the bounds specified below
nu_lower = 0
nu_upper = np.inf

# Using the commented out code below, we can also get the frequency bounds
# from the configuration -- try it out (note we convert between wavelength
# and frequency, and thus the order switches):
#nu_lower = tardis_config.supernova.luminosity_wavelength_end.to(u.Hz, u.spectral)
#nu_upper = tardis_config.supernova.luminosity_wavelength_start.to(u.Hz, u.spectral)

from tardis.spectrum.luminosity import calculate_filtered_luminosity
L_output = calculate_filtered_luminosity(transport.transport_state.emitted_packet_nu, transport.transport_state.emitted_packet_luminosity, nu_lower, nu_upper)
L_output

<Quantity 8.0276259e+42 erg / s>

In [19]:
L_requested = sim.luminosity_requested
L_requested

<Quantity 1.05927636e+43 erg / s>

In [20]:
t_inner_estimated = simulation_state.t_inner * (L_output / L_requested)**t_inner_update_exponent
t_inner_estimated

<Quantity 11411.24774714 K>

In [21]:
t_inner_updated = simulation_state.t_inner + damping_t_inner * (t_inner_estimated - simulation_state.t_inner)
t_inner_updated

<Quantity 10968.05902178 K>

We now advance the state of the simulation based on the estimators. This will also display a summary of the updated values of $T_\mathrm{rad}$ and $W$. Additionally, the ``advance_state()`` method checks for convergence and returns the convergence status as shown below.

In [22]:
sim.advance_state(emitted_luminosity=L_output)

False

Finally, we show the full updated $T_\mathrm{rad}$, $W$, and $T_\mathrm{inner}$, as well as the updated electron densities which were updated along with the rest of the plasma based on the new values for $T_\mathrm{rad}$, $W$, and $T_\mathrm{inner}$. Compare these with our calculations above and with the initial values at the beginning of the code example!

In [23]:
simulation_state.t_radiative

<Quantity [10056.09504944, 10049.64072412, 10092.17524171, 10107.09452369,
           10082.94553306, 10049.23046424, 10047.62146489, 10016.12095664,
            9989.24792574,  9981.19420201,  9944.87167476,  9905.78279726,
            9901.29983867,  9832.19729379,  9858.18339043,  9808.56786458,
            9740.59467717,  9689.54178072,  9675.80026142,  9622.17678848] K>

In [24]:
simulation_state.dilution_factor

array([0.42728118, 0.34911588, 0.29582428, 0.25735303, 0.23019053,
       0.20675476, 0.18805027, 0.17200459, 0.15791091, 0.14589747,
       0.13552955, 0.12685344, 0.11837556, 0.11205399, 0.10459577,
       0.09908548, 0.09455759, 0.09008505, 0.08536569, 0.08157118])

In [25]:
simulation_state.t_inner

<Quantity 10968.05902178 K>

In [26]:
plasma.electron_densities

0     2.878018e+09
1     2.191373e+09
2     1.686992e+09
3     1.309774e+09
4     1.025087e+09
5     8.087867e+08
6     6.433871e+08
7     5.152906e+08
8     4.155435e+08
9     3.373493e+08
10    2.754141e+08
11    2.261255e+08
12    1.867552e+08
13    1.548811e+08
14    1.292517e+08
15    1.082379e+08
16    9.101149e+07
17    7.686456e+07
18    6.521082e+07
19    5.549951e+07
dtype: float64